In [10]:
from bin.pYSAM_Solar import run_pvwatts
from models.economic import calc_ppa_price
from pathlib import Path
from capex_opex.excel_capex_opex import calc_capex_opex

weather_path = "input_data/tmy_38.730_-3.450_2005_2023.epw"
excel_file = Path("capex_opex/CAPEX OPEX model CSP-PV MJ2500.xlsx")

pv_capacity_kwdc = 30_000  # kWdc

results = run_pvwatts(weather_path,
                      out_dir="output_data",
                      system_capacity_kwdc=pv_capacity_kwdc,
                      dc_ac_ratio=1.2,
                      tilt=25.0,
                      azimuth=180.0,
                      gcr=0.4,
                      losses_percent=14.0,
                      lifetime_output=False,)
                      #module_model=None,#"SunPower SPR-E19-315",
                      #inverter_model=None,)#"Sungrow Power Supply Co - Ltd : SC2500U [550V]")

annual_kwh = results["annual_energy_kwh"]
cf = results["capacity_factor"]
df_timeseries = results["timeseries_df"]

inputs = {
    "PB Installed Capacity (Gross)": 100000,
    "Solar Field Aperture Area (Mirror Area)": 507597.5,
    "Thermal Energy Storage Capacity ": 3271.3,
    "Receiver Power (Max Rated)": 200,
    "Tower Height (w/o Receiver)": 200,
    "Electric Heater Thermal Power (Max Rated)": 200000,
    "Land Area CSP": 2_000_000,
    "PV Installed Capacity": pv_capacity_kwdc*1000,  # Wdc
    "Battery Pack Power (Max Rated)": 150,
    "Battery Pack Capacity": 300,
    "Battery Annual Generation (for OPEX)": 400,
    "Land Area PV": 1_000_000,
    "CSP Annual Generation (for OPEX)": 213.85,
    "Distance to Grid ": 0,
    "Distance to Road": 0,
    "Distance to Gas": 0,
    "Distance to Water": 0,
    "Other Component Size": 0,
}

overrides = {
    # If tower technology: SET BOP Reference to 74.8 MUSD or If PT technology: SET BOP Reference Cost to 90.2 MUSD
    "BOP": 74.8,          # MUSD (replaces CAPEX!D27)
    # If tower technology: SET Solar Field Reference Cost to 125 MUSD or If PT technology: SET Solar Field Reference Cost to 115 MUSD
    "SF_aperture": 115,  # MUSD (replaces CAPEX!D28)
    # pv_modules_ref_cost_musd=0.33,# $/Wdc
    "PV_modules": 0.3,   # MUSD (replaces CAPEX!D41)
    #Fixed Trackers	0.01 ; Single Axis Tracking	0.0175
    "pv_fixed_opex_coeff": 1,
}

capex_musd, opex_musd, capex_df, opex_df = calc_capex_opex(
    excel_file, inputs, return_breakdowns=True, ref_cost_overrides=overrides
)
print(f"Total CAPEX: {capex_musd:.3f} MUSD")
print(f"Total OPEX : {opex_musd:.3f} MUSD/year")

# After you have these:
annual_kwh = results["annual_energy_kwh"]      # Annual energy production in kWh
capex_musd = capex_musd                         # CAPEX in MUSD
opex_musd = opex_musd                           # OPEX in MUSD per year

# Convert units if needed
capex = capex_musd * 1e6                        # convert MUSD -> USD (or whichever currency) 
opex = opex_musd * 1e6                          # annual OPEX cost

# For the energy series, you may use just the annual number (simplest) or build a full timeseries array
# Here we'll assume it's constant every year (no degradation) for lifetime:
lifetime_years = 20                             # e.g., 20 years
energy_timeseries_mwh = [annual_kwh / 1000] * lifetime_years  # convert kWh -> MWh, one entry per year

# Discount rate and other assumptions
discount_rate = 0.07                             # e.g., 7%
degradation_rate = 0.0                           # assume no degradation – adjust if you have one
inflation_rate = 0.0                             # assume real terms (no inflation) – adjust if nominal

# Call PPA-price calculation
ppa_price_per_mwh = calc_ppa_price(
    energy_timeseries_mwh,
    capex,
    opex,
    lifetime_years,
    discount_rate,
    degradation_rate=degradation_rate,
    inflation_rate=inflation_rate
)

print(f"Break-even PPA price: {ppa_price_per_mwh:.2f} {''} USD per MWh")



=== PySAM PVWatts Run ===
Annual Energy: 46,300,226 kWh
Capacity Factor: 17.62 %
Saved hourly AC power to output_data\pv_ac_timeseries.csv
Total CAPEX: 659.086 MUSD
Total OPEX : 9.410 MUSD/year
Break-even PPA price: 1546.93  USD per MWh
